# 02 Muscle edits (unilateral)

Run this after `01_non_muscle_edits.ipynb`. It applies muscle-specific edits and writes final unilateral outputs.

In [7]:
import opensim as osim
from pathlib import Path
import polars as pl
import numpy as np
import sys

project_root = Path.cwd().resolve()
if project_root.name == 'pipeline':
    project_root = project_root.parent.parent
elif project_root.name == 'notebooks':
    project_root = project_root.parent

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from rathindlimb.processing import update_model, remove_muscles
from rathindlimb.muscle_utils import model_thelen_to_millard
from rathindlimb.registration import register_meshes, apply_transformation_to_mesh, convert_points_between_meshes

model_dir = project_root / 'models' / 'osim'
pipeline_dir = model_dir / '.pipeline'
data_dir = project_root / 'data'
mesh_dir = project_root / 'models' / 'meshes'
attachment_dir = data_dir / 'attachments'

input_file = pipeline_dir / 'rat_hindlimb_non_muscle.osim'
unilateral_out = model_dir / 'rat_hindlimb_unilateral.osim'
unilateral_no_muscles_out = model_dir / 'rat_hindlimb_unilateral_no_muscles.osim'

model = osim.Model(str(input_file))
model = model_thelen_to_millard(model)

[info] Loaded model RatHindlimbRight from file /home/hudson/RatHindlimb/models/osim/.pipeline/rat_hindlimb_non_muscle.osim
[warning] Couldn't find file 'ground.stl'.
[warning] Couldn't find file 'spine.stl'.
[warning] Couldn't find file 'pelvis_r.stl'.
[warning] Couldn't find file 'femur_r.stl'.
[warning] Couldn't find file 'tibia_r.stl'.
[warning] Couldn't find file 'foot_r.stl'.


In [8]:
source_file = mesh_dir / 'foot3_r_young_no_phalanges.stl'
target_file = mesh_dir / 'foot2_r_young.stl'
output_file1 = mesh_dir / 'foot_r_young_no_phalanges.stl'
foot_file = mesh_dir / 'foot3_r_young.stl'
final_file = mesh_dir / 'foot_r_young.stl'
transform = register_meshes(source_file, target_file, output_file1, seed=123)
apply_transformation_to_mesh(foot_file, transform, final_file)

source_postfix = '_young.stl'
target_postfix = '_johnson.stl'
output_postfix = '.stl'
output_dir = model_dir / 'Geometry'
meshes = ['spine', 'pelvis_r', 'femur_r', 'tibia_r', 'foot_r']
transforms = {}
for mesh in meshes:
    source_file = mesh_dir / (mesh + source_postfix)
    target_file = mesh_dir / (mesh + target_postfix)
    output_file = output_dir / (mesh + output_postfix)
    transforms[mesh] = register_meshes(source_file, target_file, output_file, seed=120)

young_attachments = pl.read_csv(attachment_dir / 'young_model_attachments.csv')
young_attachments = young_attachments.with_columns([
    pl.col('X (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
    pl.col('Y (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
    pl.col('Z (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64)
])

Loading and preparing meshes...
Preprocessing point clouds and computing features...
Performing global registration (RANSAC)...
[Open3D WARNING] Too few correspondences (130) after mutual filter, fall back to original correspondences.
Refining registration (ICP)...

Registered mesh saved to /home/hudson/RatHindlimb/models/meshes/foot_r_young_no_phalanges.stl
Transformed mesh saved to /home/hudson/RatHindlimb/models/meshes/foot_r_young.stl
Loading and preparing meshes...
Preprocessing point clouds and computing features...
Performing global registration (RANSAC)...
Refining registration (ICP)...

Registered mesh saved to /home/hudson/RatHindlimb/models/osim/Geometry/spine.stl
Loading and preparing meshes...
Preprocessing point clouds and computing features...
Performing global registration (RANSAC)...
Refining registration (ICP)...

Registered mesh saved to /home/hudson/RatHindlimb/models/osim/Geometry/pelvis_r.stl
Loading and preparing meshes...
Preprocessing point clouds and computing

In [9]:
body_set = model.getBodySet()
for i in range(body_set.getSize()):
    body = body_set.get(i)
    body.upd_WrapObjectSet().clearAndDestroy()

muscles = model.getMuscles()
for i in range(muscles.getSize()):
    muscle = muscles.get(i)
    muscle_name = muscle.getName()
    geo_path = muscle.getGeometryPath()
    path_points = geo_path.getPathPointSet()
    existing_path_points = path_points.getSize()
    geo_path.getWrapSet().clearAndDestroy()

    rows = young_attachments.filter(pl.col('Abbreviation') == muscle_name.split('R_')[-1])
    index = 1
    for row in rows.iter_rows(named=True):
        lower_frame = row['Frame'].lower()
        frame = lower_frame + '_r' if lower_frame != 'spine' else lower_frame
        mesh = model.getBodySet().get(frame)
        point_name = muscle_name + '_' + frame + '_' + row['Type'].lower() + '_' + str(index)
        young_loc = np.array([[row['X (mm)'], row['Y (mm)'], row['Z (mm)']]]) / 100
        loc = convert_points_between_meshes(young_loc, transforms[frame])
        vec = osim.Vec3(loc[0][0], loc[0][1], loc[0][2])
        muscle.addNewPathPoint(point_name, mesh, vec)
        index += 1

    for j in range(existing_path_points - 1, -1, -1):
        path_points.remove(j)

In [10]:
parameter_dir = data_dir / 'parameters'
johnson_params = pl.read_csv(parameter_dir / 'johnson_2011_parameters.csv')
tsl_df = pl.read_csv(parameter_dir / 'tsl_comparison.csv')

muscles = model.getMuscles()
for i in range(muscles.getSize()):
    muscle = muscles.get(i)
    muscle_name = muscle.getName().replace('R_', '')

    params = johnson_params.row(by_predicate=pl.col('Abbreviation') == muscle_name, named=True)
    if params is not None:
        muscle.setMaxIsometricForce(params['Fo (N)'])
        muscle.setOptimalFiberLength(params['l0 (mm)'] / 1000)
        muscle.setPennationAngleAtOptimalFiberLength(params['θ0 (deg)'] * np.pi / 180)

    tsl_params = tsl_df.row(by_predicate=pl.col('Abbreviation') == muscle_name, named=True)
    if tsl_params is not None:
        lts = tsl_params['Walk TSL (mm)'] / 1000
        if lts <= 0:
            muscle.set_ignore_tendon_compliance(True)
        muscle.setTendonSlackLength(lts)

model = update_model(model, unilateral_out)

no_muscles_model = osim.Model(str(unilateral_out))
no_muscles_model = remove_muscles(no_muscles_model)
_ = update_model(no_muscles_model, unilateral_no_muscles_out)

print(f'Wrote {unilateral_out}')
print(f'Wrote {unilateral_no_muscles_out}')

[info] 'R_BFa' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_BFp' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_CF' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_STp' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_STa' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_SM' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_QF' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_TFL' Parameter update for the damped-model: ActiveForceLengthCurve minimum value was 0.1 but is now 0.0.
[info] 'R_GMa' Parameter update for the damped-model: ActiveForceLengthCurve minimu